# SEG mặt đường — thử 3 model, chọn 1 model cho mỗi bài

**Một notebook cho cả crack và pothole; tất cả hàm nằm trong file này.** Không cần import mã từ `src/road_damage`.

Luồng giống notebook DET: **kiểm tra dữ liệu → xem nhãn → thử YOLOv8n/s/m-seg trong 30 epoch → so sánh mAP50, mAP50–95, P, R và thời gian → chọn một model cho từng dataset → train model được chọn 100 epoch → đánh giá test và đo mask.**

Chọn kernel **Road damage SEG (Python 3.11 CUDA)**. Các lượt pilot đang chạy nền được đọc lại, không khởi chạy trùng. Vòng 100 epoch chỉ chạy khi bạn bật `RUN_MAIN_100` ở ô cấu hình.

In [ ]:
from pathlib import Path
from contextlib import contextmanager
from datetime import datetime, timezone
from collections import Counter
from html import escape
import os, sys, json, time, math, random, hashlib, gc, io, base64

import psutil
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, FileLink

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data" / "raw").is_dir() and (p / "models" / "pretrained").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Mở notebook từ thư mục Hu_hong_duong hoặc notebooks.")
runtime = ROOT / "reports" / "runtime" / "ultralytics"
runtime.mkdir(parents=True, exist_ok=True)
os.environ["YOLO_CONFIG_DIR"] = str(runtime)

import cv2
import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.models.yolo.segment import SegmentationTrainer
from skimage.morphology import skeletonize

print("Project:", ROOT)
print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "| Ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Chưa có CUDA")
if not torch.cuda.is_available():
    raise RuntimeError("Chọn kernel Road damage SEG (Python 3.11 CUDA), rồi chạy lại notebook.")
ENVIRONMENT = dict(python=sys.version.split()[0], torch=torch.__version__,
                   cuda_runtime=torch.version.cuda, ultralytics=ultralytics.__version__,
                   gpu=torch.cuda.get_device_name(0))
print("CUDA đã sẵn sàng trên cuda:0.")


## 1. Cấu hình chung

Hai bài dùng nhãn gốc: `crack` để đo chiều dài, `Pothole` để đo diện tích. Chọn model riêng từng bài, không so mAP giữa hai bộ dữ liệu như cùng một bài toán.

In [ ]:
DATASETS = {
    "crack": ROOT / "data" / "raw" / "crack-4",
    "pothole": ROOT / "data" / "raw" / "pothole-v2",
}
MODEL_SIZES = ("n", "s", "m")
BATCH = {"n": 8, "s": 8, "m": 4}
PILOT_EPOCHS = 30
MAIN_EPOCHS = 100
IMGSZ, SEED = 640, 42
ACCURACY_GAP = 0.02             # Cho phép kém mAP50–95 tốt nhất tối đa 2 điểm %
RUN_MAIN_100 = True              # Pilot đã đủ 6/6; tiếp tục hai model được đề xuất
BENCHMARK_IMAGES, BENCHMARK_REPEATS, WARMUP = 200, 3, 10

# Kết quả mới do notebook chạy nằm riêng; kết quả pilot đang chạy được đọc lại.
NOTEBOOK_RUNS = ROOT / "runs" / "seg" / "single_notebook"
EXISTING_PILOT = ROOT / "runs" / "seg" / "pilot30"
EVALUATION_FILE = ROOT / "reports" / "seg_notebook_evaluation.html"

COMMON_TRAIN = dict(
    task="segment", imgsz=IMGSZ, device=0, workers=0, amp=True, cache=False,
    seed=SEED, deterministic=True, optimizer="AdamW", lr0=0.001, lrf=0.01,
    momentum=0.937, weight_decay=0.0005, warmup_epochs=3.0,
    warmup_momentum=0.8, warmup_bias_lr=0.1, box=7.5, cls=0.5, dfl=1.5,
    nbs=64, cos_lr=False, close_mosaic=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=0.0, translate=0.1, scale=0.5, shear=0.0, perspective=0.0,
    flipud=0.0, fliplr=0.5, mosaic=1.0, mixup=0.0, copy_paste=0.0,
    rect=False, multi_scale=0.0, single_cls=False, overlap_mask=True,
    mask_ratio=4, fraction=1.0, pretrained=True, freeze=None, val=True,
    save=True, save_period=10, plots=True, profile=False, compile=False,
)

## 2. Hàm dùng chung và tránh chạy chồng GPU

Các ô bên dưới chỉ định nghĩa hàm. Khi hàng đợi pilot cũ đang chạy, ô train của notebook hiển thị tiến độ và để hàng đợi tiếp tục.

In [ ]:
def read_json(path, default=None):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8-sig")) if path.exists() else default

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.{os.getpid()}.tmp")
    tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    tmp.replace(path)

def now():
    return datetime.now(timezone.utc).isoformat()

def process_alive(pid, created=None, expected_script=None):
    try:
        p = psutil.Process(int(pid))
        if created is not None and abs(p.create_time() - created) >= 1:
            return False
        if expected_script and not any(expected_script in arg for arg in p.cmdline()):
            return False
        return p.is_running()
    except (psutil.Error, ValueError, TypeError):
        return False

def background_job():
    notebook_job = read_json(ROOT / "reports/runtime/notebook_execution.json", {})
    if (notebook_job.get("pid") != os.getpid()
            and process_alive(notebook_job.get("pid"),notebook_job.get("process_created"))):
        return f"Notebook đang chạy {notebook_job.get('stage','SEG')} (PID {notebook_job['pid']})"
    queue = read_json(ROOT / "reports/runtime/queue_process.json", {})
    if process_alive(queue.get("pid"), queue.get("process_created")):
        return f"Hàng đợi pilot đang chạy (PID {queue['pid']})"
    final = read_json(ROOT / "reports/runtime/finalization.json", {})
    if process_alive(final.get("pid"), expected_script="finalize_pilot.py"):
        return f"Tiến trình hoàn thiện pilot đang chạy (PID {final['pid']})"
    # Finalizer có thể bỏ PID khỏi JSON trong giai đoạn retry.
    for proc in psutil.process_iter(["pid", "cmdline", "exe"]):
        try:
            args = proc.info.get("cmdline") or []
            is_finalizer = any(Path(arg).name.lower() == "finalize_pilot.py" for arg in args)
            belongs_here = (Path(proc.info.get("exe") or "").resolve() == (ROOT / ".conda/python.exe").resolve()
                            or any(str(ROOT).casefold() in arg.casefold() for arg in args))
            if is_finalizer and belongs_here and proc.is_running():
                return f"Tiến trình hoàn thiện pilot đang chạy (PID {proc.pid})"
        except (psutil.Error, OSError):
            continue
    lock = read_json(ROOT / "runs/seg/.gpu.lock", {})
    for pid_key, time_key in (("pid", "created"), ("worker_pid", "worker_created")):
        if process_alive(lock.get(pid_key), lock.get(time_key)):
            return f"GPU đang được dùng bởi PID {lock[pid_key]}"
    return None

@contextmanager
def exclusive_gpu():
    owner = background_job()
    if owner:
        raise RuntimeError(owner)
    path = ROOT / "runs/seg/.gpu.lock"
    path.parent.mkdir(parents=True, exist_ok=True)
    token = f"{os.getpid()}:{time.time_ns()}"
    # Chỉ một notebook được xử lý khóa cũ tại một thời điểm.
    guard = (path.parent / ".notebook_guard").open("a+b")
    if os.name == "nt":
        import msvcrt
        if guard.seek(0,2) == 0:
            guard.write(b"0"); guard.flush()
        guard.seek(0)
        try:
            msvcrt.locking(guard.fileno(),msvcrt.LK_NBLCK,1)
        except OSError:
            guard.close()
            raise RuntimeError("Một notebook khác đang giữ GPU.") from None
    else:
        import fcntl
        try:
            fcntl.flock(guard,fcntl.LOCK_EX | fcntl.LOCK_NB)
        except OSError:
            guard.close()
            raise RuntimeError("Một notebook khác đang giữ GPU.") from None
    try:
        owner = background_job()
        if owner:
            raise RuntimeError(owner)
        for attempt in range(3):
            try:
                with path.open("x",encoding="utf-8") as stream:
                    json.dump({"pid":os.getpid(),"created":psutil.Process().create_time(),"token":token},stream)
                break
            except FileExistsError:
                try:
                    previous = read_json(path,{})
                except json.JSONDecodeError:
                    raise RuntimeError("GPU lock đang được ghi; thử lại sau.") from None
                if any(process_alive(previous.get(pk),previous.get(tk))
                       for pk,tk in (("pid","created"),("worker_pid","worker_created"))):
                    raise RuntimeError("Tiến trình khác vừa lấy GPU; không chạy chồng.")
                path.unlink(missing_ok=True)
        else:
            raise RuntimeError("Chưa lấy được GPU lock.")
        try:
            yield
        finally:
            if read_json(path,{}).get("token") == token:
                path.unlink(missing_ok=True)
    finally:
        if os.name == "nt":
            guard.seek(0)
            msvcrt.locking(guard.fileno(),msvcrt.LK_UNLCK,1)
        else:
            fcntl.flock(guard,fcntl.LOCK_UN)
        guard.close()


def file_sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream,"sha256").hexdigest()


def evaluation_signature(checkpoint,dataset):
    payload = dict(checkpoint=file_sha256(checkpoint),data=DATA_SIGNATURES[dataset],
                   sample=SAMPLES[dataset]["sha256"],imgsz=IMGSZ,batch=1,half=True,rect=False,
                   validation_conf=.001,predict_conf=.25,iou=.7,max_det=300,retina_masks=True,
                   repeats=BENCHMARK_REPEATS,warmup=WARMUP,environment=ENVIRONMENT)
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()


def model_name(size):
    return f"yolov8{size}-seg"

def own_run(dataset, size, stage):
    return NOTEBOOK_RUNS / stage / dataset / f"yolov8{size}_seg"

def label_path(image):
    parts = list(Path(image).parts)
    index = max(i for i, part in enumerate(parts) if part.casefold() == "images")
    parts[index] = "labels"
    return Path(*parts).with_suffix(".txt")

def manifest_images(data_yaml, split):
    cfg = yaml.safe_load(Path(data_yaml).read_text(encoding="utf-8-sig"))
    base = Path(cfg.get("path", Path(data_yaml).parent))
    entries = cfg[split] if isinstance(cfg[split], list) else [cfg[split]]
    images = []
    for entry in entries:
        manifest = Path(entry)
        if not manifest.is_absolute():
            manifest = base / manifest
        for line in manifest.read_text(encoding="utf-8-sig").splitlines():
            if line.strip():
                image = Path(line.strip())
                images.append((image if image.is_absolute() else manifest.parent / image).resolve())
    return sorted(set(images), key=lambda p: str(p).lower())

def data_signature(data_yaml):
    # Cùng quy tắc với các lượt pilot đã chạy để kiểm tra khả năng dùng lại.
    path = Path(data_yaml)
    cfg = yaml.safe_load(path.read_text(encoding="utf-8"))
    base = Path(cfg.get("path", path.parent))
    digest = hashlib.sha256(path.read_bytes())
    for split in ("train", "val", "test"):
        entries = cfg[split] if isinstance(cfg[split], list) else [cfg[split]]
        for entry in entries:
            manifest = Path(entry)
            if not manifest.is_absolute():
                manifest = base / manifest
            digest.update(manifest.read_bytes())
            for line in manifest.read_text(encoding="utf-8-sig").splitlines():
                if line.strip():
                    image = Path(line.strip())
                    if not image.is_absolute():
                        image = manifest.parent / image
                    stat = image.stat()
                    digest.update(f"{image}:{stat.st_size}:{stat.st_mtime_ns}".encode("utf-8"))
                    digest.update(label_path(image).read_bytes())
    return digest.hexdigest()

def benchmark_sample(data_yaml):
    images = manifest_images(data_yaml, "val")
    indices = sorted(random.Random(SEED).sample(range(len(images)), min(BENCHMARK_IMAGES, len(images))))
    chosen = [str(images[i]) for i in indices]
    return {"images": chosen, "sha256": hashlib.sha256("\n".join(chosen).encode("utf-8")).hexdigest()}

## 3. Kiểm tra dữ liệu cơ bản

Kiểm tra ảnh, cặp ảnh–nhãn, polygon và class. Giữ TXT rỗng hợp lệ là ảnh nền; ảnh thiếu nhãn/lỗi được loại khỏi danh sách dùng, không sửa dữ liệu gốc. Khi có audit đã chạy, notebook đọc lại kết quả và giữ nguyên YAML/manifests của hàng đợi.

In [ ]:
def parse_polygons(text, nc):
    polygons = []
    for line in text.splitlines():
        parts = line.split()
        if not parts:
            continue
        cls = int(parts[0])
        if not 0 <= cls < nc or len(parts) < 7 or (len(parts) - 1) % 2:
            raise ValueError("Class hoặc số đỉnh polygon không hợp lệ")
        points = np.asarray(parts[1:], dtype=float).reshape(-1, 2)
        if not np.isfinite(points).all() or (points < 0).any() or (points > 1).any():
            raise ValueError("Tọa độ polygon phải hữu hạn, thuộc [0,1]")
        if len(np.unique(points, axis=0)) < 3:
            raise ValueError("Polygon cần ít nhất 3 đỉnh phân biệt")
        polygons.append((cls, points))
    return polygons

def check_data(dataset):
    cached_path = ROOT / f"reports/data_quality/{dataset}/audit.json"
    cached = read_json(cached_path)
    if cached and cached.get("status") == "ready" and Path(cached["yaml_path"]).is_file():
        print(dataset, "— dùng audit đã kiểm tra:", cached_path.relative_to(ROOT))
        return cached
    if background_job():
        raise RuntimeError("Chờ hàng đợi kết thúc trước khi tạo lại cấu hình dữ liệu.")
    raw = DATASETS[dataset]
    original = yaml.safe_load((raw / "data.yaml").read_text(encoding="utf-8-sig"))
    names = original["names"]
    names = dict(enumerate(names)) if isinstance(names, list) else {int(k):v for k,v in names.items()}
    if sorted(names) != list(range(len(names))):
        raise ValueError("Class ID cần liên tiếp từ 0")
    audit = {"dataset": dataset, "class_names": names, "splits": {}, "excluded": []}
    generated = ROOT / "data/manifests/single_notebook" / dataset
    generated.mkdir(parents=True, exist_ok=True)
    cfg = {"path": str(ROOT), "nc": len(names), "names": names}
    for split, folder in (("train", "train"), ("val", "valid"), ("test", "test")):
        directory = raw / folder
        if not directory.exists() and split == "val":
            directory = raw / "val"
        images = sorted(p for p in (directory / "images").iterdir()
                        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"})
        counts = Counter(p.stem.casefold() for p in images)
        kept, blanks, polygons = [], 0, 0
        for image in images:
            try:
                if counts[image.stem.casefold()] != 1:
                    raise ValueError("Trùng tên ảnh, không xác định duy nhất nhãn")
                with Image.open(image) as picture:
                    picture.load()
                rows = parse_polygons(label_path(image).read_text(encoding="utf-8-sig"), len(names))
                kept.append(image.resolve())
                blanks += not rows
                polygons += len(rows)
            except (OSError, ValueError) as error:
                audit["excluded"].append({"split": split, "image": str(image), "reason": str(error)})
        if not kept:
            raise ValueError(f"{dataset}/{split}: không có ảnh hợp lệ")
        manifest = generated / f"{split}.txt"
        manifest.write_text("\n".join(p.as_posix() for p in kept) + "\n", encoding="utf-8")
        cfg[split] = manifest.as_posix()
        audit["splits"][split] = {"total_images": len(images), "usable_images": len(kept),
                                  "excluded_images": len(images)-len(kept), "blank_labels": blanks,
                                  "polygon_count": polygons}
    config_path = ROOT / "configs/seg/single_notebook" / f"{dataset}.yaml"
    config_path.parent.mkdir(parents=True, exist_ok=True)
    config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    audit.update(status="ready", yaml_path=str(config_path))
    write_json(ROOT / f"reports/single_notebook/{dataset}_audit.json", audit)
    return audit

In [ ]:
AUDITS = {dataset: check_data(dataset) for dataset in DATASETS}
DATA_YAMLS = {dataset: Path(audit["yaml_path"]) for dataset, audit in AUDITS.items()}
DATA_SIGNATURES = {dataset: data_signature(path) for dataset, path in DATA_YAMLS.items()}
SAMPLES = {dataset: benchmark_sample(path) for dataset, path in DATA_YAMLS.items()}

rows = []
for dataset, audit in AUDITS.items():
    print(dataset, "— classes:", audit["class_names"])
    for split, values in audit["splits"].items():
        rows.append({"Dataset": dataset, "Split": split,
                     "Ảnh gốc": values["total_images"], "Ảnh sử dụng": values["usable_images"],
                     "Ảnh loại": values["excluded_images"], "Ảnh nền": values["blank_labels"],
                     "Polygon": values["polygon_count"]})
display(pd.DataFrame(rows))

In [ ]:
for dataset in DATASETS:
    images = manifest_images(DATA_YAMLS[dataset], "train")
    selected = [images[i] for i in np.linspace(0, len(images)-1, 6, dtype=int)]
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for ax, path in zip(axes.flat, selected):
        with Image.open(path) as source:
            image = np.array(source.convert("RGB"))
        ax.imshow(image)
        for cls, points in parse_polygons(label_path(path).read_text(encoding="utf-8-sig"),
                                          len(AUDITS[dataset]["class_names"])):
            xy = points * np.array([image.shape[1]-1, image.shape[0]-1])
            ax.fill(xy[:,0], xy[:,1], color="lime", alpha=.2)
            ax.plot(*np.vstack([xy, xy[0]]).T, color="lime", linewidth=1)
        ax.axis("off")
    fig.suptitle(f"{dataset}: kiểm tra polygon trên 6 ảnh train")
    plt.tight_layout()
    plt.show()

## 4. Hàm huấn luyện, đánh giá và đo thời gian

Checkpoint tốt nhất chọn theo **mask mAP50–95 validation**. Pilot đủ 30 epoch, không dừng sớm. Vòng chính bắt đầu lại từ pretrained của model được chọn, tối đa 100 epoch và patience 20; không resume trực tiếp một pilot đã hoàn tất sang lịch 100 epoch.

Benchmark dùng cùng tối đa 200 ảnh validation, 10 lượt warm-up, 3 lần lặp, batch 1, FP16 và đồng bộ CUDA. Thời gian đọc ảnh khỏi đĩa nằm ngoài vùng đo.

In [ ]:
class MaskTrainer(SegmentationTrainer):
    final_eval_seconds = 0.0
    def validate(self):
        old_best = self.best_fitness
        metrics, _ = super().validate()
        if metrics is None:
            return None, None
        score = float(metrics["metrics/mAP50-95(M)"])
        self.best_fitness = max(old_best if old_best is not None else -math.inf, score)
        return metrics, score

    def final_eval(self):
        start = time.perf_counter()
        try:
            return super().final_eval()
        finally:
            self.final_eval_seconds += time.perf_counter() - start

def evaluate_best(checkpoint, data_yaml, output, split="val"):
    model = YOLO(str(checkpoint), task="segment")
    metrics = model.val(
        data=str(data_yaml), split=split, imgsz=IMGSZ, batch=1, device=0,
        workers=0, half=True, rect=False, conf=.001, iou=.7, max_det=300,
        plots=True, save_json=False, project=str(output.parent), name=output.name,
        exist_ok=True, verbose=False,
    )
    def summary(metric):
        values = dict(precision=float(metric.mp), recall=float(metric.mr),
                      map50=float(metric.map50), map50_95=float(metric.map))
        if not all(math.isfinite(v) and 0 <= v <= 1 for v in values.values()):
            raise ValueError("Metrics không hợp lệ")
        return values
    result = {"split": split, "mask": summary(metrics.seg), "box": summary(metrics.box),
              "checkpoint": str(checkpoint), "checkpoint_selection": "validation_mask_map50_95"}
    del model, metrics
    gc.collect()
    torch.cuda.empty_cache()
    return result

def benchmark_model(checkpoint, sample):
    images = [cv2.imread(path) for path in sample["images"]]
    if not images or any(image is None for image in images):
        raise ValueError("Không đọc được ảnh benchmark")
    model = YOLO(str(checkpoint), task="segment")
    args = dict(imgsz=IMGSZ, batch=1, device=0, half=True, rect=False,
                conf=.25, iou=.7, max_det=300, retina_masks=True,
                verbose=False, save=False)
    for i in range(WARMUP):
        model.predict(images[i % len(images)], **args)
    timings = []
    for _ in range(BENCHMARK_REPEATS):
        for image in images:
            torch.cuda.synchronize(0)
            start = time.perf_counter()
            result = model.predict(image, **args)[0]
            torch.cuda.synchronize(0)
            timings.append((time.perf_counter()-start)*1000)
            if result.masks is not None and tuple(result.masks.data.shape[-2:]) != tuple(result.orig_shape):
                raise ValueError("Mask không khớp kích thước ảnh gốc")
    mean = float(np.mean(timings))
    report = dict(mean_ms=mean, median_ms=float(np.median(timings)),
                  p95_ms=float(np.percentile(timings,95)), fps=1000/mean,
                  manifest_sha256=sample["sha256"], images=len(images),
                  repeats=BENCHMARK_REPEATS, warmup=WARMUP, args=args,
                  scope="end_to_end_predict_excluding_disk", gpu=torch.cuda.get_device_name(0))
    del model, images, result
    gc.collect()
    torch.cuda.empty_cache()
    return report

def training_compatible(report):
    """Nhận diện training hoàn tất riêng với cấu hình benchmark."""
    try:
        cfg = report["config"]
        checkpoint = report["checkpoint"]
        size = next(s for s in MODEL_SIZES if model_name(s) == report["model"])
        return (report["status"] == "completed"
                and report["epochs_completed"] == report["epochs_requested"] == PILOT_EPOCHS
                and cfg.get("epochs") == PILOT_EPOCHS and cfg.get("patience") == 0
                and all(cfg.get(k) == v for k,v in COMMON_TRAIN.items())
                and cfg.get("batch") == BATCH[size]
                and math.isfinite(report["train_wall_seconds"]) and report["train_wall_seconds"] > 0
                and all(report.get("environment",{}).get(k) == v for k,v in ENVIRONMENT.items())
                and report.get("dataset_fingerprint") == DATA_SIGNATURES[report["dataset"]]
                and Path(checkpoint["best"]).is_file() and Path(checkpoint["last"]).is_file())
    except (KeyError,TypeError,ValueError,StopIteration):
        return False


def complete_pilot(report):
    try:
        mask, bench = report["validation"]["mask"], report["benchmark"]
        expected_bench = dict(imgsz=IMGSZ,batch=1,half=True,rect=False,conf=.25,iou=.7,
                              max_det=300,retina_masks=True,device=0)
        return (training_compatible(report)
                and report["validation"]["split"] == "val"
                and all(math.isfinite(mask[k]) and 0 <= mask[k] <= 1
                        for k in ("precision","recall","map50","map50_95"))
                and all(math.isfinite(bench[k]) and bench[k] > 0
                        for k in ("mean_ms","median_ms","p95_ms","fps"))
                and all(bench.get("args",{}).get(k) == v for k,v in expected_bench.items())
                and bench.get("repeats") == BENCHMARK_REPEATS and bench.get("warmup") == WARMUP
                and bench.get("manifest_sha256") == SAMPLES[report["dataset"]]["sha256"])
    except (KeyError,TypeError,ValueError):
        return False


def get_pilot(dataset, size):
    paths = [own_run(dataset,size,f"pilot{PILOT_EPOCHS}") / "report.json"]
    if PILOT_EPOCHS == 30:
        paths.append(EXISTING_PILOT / dataset / f"yolov8{size}_seg" / "report.json")
    found = []
    for path in paths:
        report = read_json(path)
        if report:
            report = {**report, "_source": str(path)}
            found.append(report)
            if complete_pilot(report):
                return report
    return found[0] if found else {"dataset":dataset, "model":model_name(size),
                                   "status":"pending", "epochs_completed":0}

In [ ]:
def train_one(dataset, size, stage):
    pilot = stage.startswith("pilot")
    epochs = PILOT_EPOCHS if pilot else MAIN_EPOCHS
    if pilot:
        previous = get_pilot(dataset,size)
        if complete_pilot(previous):
            print("Dùng lại pilot hoàn tất:", dataset, model_name(size))
            return previous
    run = own_run(dataset,size,stage)
    report_path = run / "report.json"
    cfg = {**COMMON_TRAIN, "data":str(DATA_YAMLS[dataset]), "batch":BATCH[size],
           "epochs":epochs, "patience":0 if pilot else 20,
           "model":str(ROOT / "models/pretrained" / f"{model_name(size)}.pt"),
           "project":str(run.parent), "name":run.name, "exist_ok":True}
    source_sha = file_sha256(cfg["model"])
    identity = dict(config=cfg,data=DATA_SIGNATURES[dataset],pretrained_sha256=source_sha,
                    environment=ENVIRONMENT)
    signature = hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()
    record = read_json(report_path)
    if record and record.get("request_signature") != signature:
        raise RuntimeError(f"Cấu hình/dữ liệu/môi trường thay đổi: dùng thư mục run mới, không ghi đè {run}")
    source_checkpoint = (record or {}).get("training_source_checkpoint",{})
    best = Path(source_checkpoint.get("best",run / "weights/best.pt"))
    last = Path(source_checkpoint.get("last",run / "weights/last.pt"))
    if record and record.get("status") == "completed" and best.exists():
        evaluation_matches = record.get("evaluation_signature") == evaluation_signature(best,dataset)
        if evaluation_matches and (complete_pilot(record) if pilot else bool(record.get("test"))):
            print("Dùng lại lượt hoàn tất:",dataset,model_name(size),f"{record['epochs_completed']}/{epochs} epoch")
            return {**record, "_source":str(report_path)}
        print("Dùng checkpoint đã train; cập nhật đánh giá/benchmark:",dataset,model_name(size))
    if record is None:
        if best.exists() or last.exists() or (run / "results.csv").exists():
            raise RuntimeError("Có kết quả không có report; cần kiểm tra trước khi train lại.")
        record = dict(dataset=dataset, model=model_name(size), stage=stage, status="queued",
                      config=cfg, request_signature=signature, dataset_fingerprint=DATA_SIGNATURES[dataset],
                      environment=ENVIRONMENT, pretrained_sha256=source_sha, worker_pid=os.getpid(),
                      epochs_requested=epochs, epochs_completed=0, train_wall_seconds=0,
                      batch=BATCH[size], created_at=now())
        if pilot and training_compatible(previous):
            # Chỉ protocol đo thay đổi: dùng trọng số pilot cũ, không train lại.
            record.update(training_finished=True,epochs_completed=PILOT_EPOCHS,
                          train_wall_seconds=previous["train_wall_seconds"],
                          training_source_run=str(Path(previous["_source"]).parent),
                          training_source_checkpoint=previous["checkpoint"])
            best,last = Path(previous["checkpoint"]["best"]),Path(previous["checkpoint"]["last"])
            print("Đánh giá lại checkpoint pilot hiện có:",best)
    with exclusive_gpu():
        if not torch.cuda.is_available():
            raise RuntimeError("Cần CUDA để train và so sánh tốc độ cùng điều kiện.")
        csv_path = Path(record.get("training_source_run",run)) / "results.csv"
        observed = pd.read_csv(csv_path)["epoch"].astype(int).tolist() if csv_path.exists() else []
        if observed != list(range(1,len(observed)+1)) or len(observed) > epochs:
            raise RuntimeError("Lịch sử epoch không liên tục/không khớp cấu hình; không tự resume.")
        training_done = best.exists() and last.exists() and (
            bool(record.get("training_finished")) or len(observed) == epochs)
        if last.exists() and not training_done:
            # Checkpoint trong thư mục run do notebook tạo, không nạp file lạ.
            state = torch.load(last,map_location="cpu",weights_only=False)
            saved_epoch = int(state.get("epoch",-1)) + 1
            stripped = saved_epoch == 0 and state.get("optimizer") is None
            del state
            if stripped:
                if pilot or not best.exists() or not observed:
                    raise RuntimeError("Checkpoint đã kết thúc nhưng thiếu số epoch yêu cầu.")
                training_done = True  # Vòng chính đã early-stop trước khi kịp ghi report.
            elif saved_epoch != len(observed) or record["epochs_completed"] not in (saved_epoch,saved_epoch-1):
                raise RuntimeError("Epoch checkpoint, CSV và report không khớp; cần kiểm tra trước khi resume.")
        if not training_done:
            if record["epochs_completed"] and not last.exists():
                raise RuntimeError("Thiếu last.pt của lượt đang dở; không tự train lại từ đầu.")
            resume = last.exists()
            model = YOLO(str(last if resume else cfg["model"]), task="segment")
            prior_time = float(record.get("train_wall_seconds",0))
            start = time.perf_counter()
            record.update(status="running", phase="training", worker_pid=os.getpid(), updated_at=now())
            write_json(report_path,record)
            def save_progress(trainer):
                record.update(epochs_completed=int(trainer.epoch)+1,
                              train_wall_seconds=prior_time+time.perf_counter()-start,
                              batch=int(trainer.batch_size), updated_at=now())
                write_json(report_path,record)
            model.add_callback("on_model_save",save_progress)
            try:
                if resume:
                    model.train(trainer=MaskTrainer, resume=True)
                else:
                    model.train(trainer=MaskTrainer, **cfg)
                record["training_finished"] = True
            except Exception as error:
                record.update(status="failed", error=str(error))
                raise
            finally:
                elapsed = time.perf_counter()-start-getattr(getattr(model,"trainer",None),"final_eval_seconds",0)
                record["train_wall_seconds"] = prior_time+elapsed
                write_json(report_path,record)
            del model
            gc.collect()
            torch.cuda.empty_cache()
        if not best.exists():
            raise RuntimeError("Không tìm thấy best.pt")
        epochs_observed = pd.read_csv(csv_path)["epoch"].astype(int).tolist()
        if not epochs_observed or (pilot and epochs_observed != list(range(1,epochs+1))):
            raise RuntimeError(f"Pilot cần đủ {epochs} epoch, thực tế: {epochs_observed}")
        record.update(epochs_completed=max(epochs_observed), training_finished=True,
                      status="running", phase="evaluation", error=None,
                      checkpoint={"best":str(best),"last":str(last),"best_sha256":file_sha256(best)}, updated_at=now())
        write_json(report_path,record)
        try:
            record["validation"] = evaluate_best(best, DATA_YAMLS[dataset], run / "evaluation/val")
            if not pilot:
                record["test"] = evaluate_best(best, DATA_YAMLS[dataset], run / "evaluation/test", split="test")
            record["benchmark"] = benchmark_model(best,SAMPLES[dataset])
            record.update(status="completed",phase="completed",completed_at=now(),updated_at=now(),
                          evaluation_signature=evaluation_signature(best,dataset))
        except Exception as error:
            record.update(status="failed",error=str(error),updated_at=now())
            raise
        finally:
            write_json(report_path,record)
    return {**record, "_source":str(report_path)}


## 5. Thử ba model — mỗi model 30 epoch

Ô này xử lý cả hai bộ theo thứ tự crack → pothole, mỗi bộ n → s → m. Lượt hoàn tất được dùng lại. Nếu hàng đợi cũ còn hoạt động, chỉ hiển thị tiến độ hiện có; không mở một lượt GPU khác.

In [ ]:
owner = background_job()
if owner:
    print(owner)
    print("Đọc kết quả pilot đã lưu; tiến trình nền tiếp tục chạy.")
else:
    for dataset in DATASETS:
        for size in MODEL_SIZES:
            print(f"\n{'='*60}\nPILOT {dataset} / {model_name(size)} / {PILOT_EPOCHS} epoch")
            train_one(dataset,size,f"pilot{PILOT_EPOCHS}")

PILOT_REPORTS = [get_pilot(dataset,size) for dataset in DATASETS for size in MODEL_SIZES]
display(pd.DataFrame([{"Dataset":r["dataset"],"Model":r["model"],"Trạng thái":r["status"],
                      "Epoch":f"{r.get('epochs_completed',0)}/{PILOT_EPOCHS}"} for r in PILOT_REPORTS]))

## 6. So sánh và chọn một model cho mỗi bài

Ưu tiên mask mAP50–95; trong nhóm cách điểm tốt nhất không quá **2 điểm phần trăm**, chọn model suy luận nhanh nhất. Nếu bằng nhau, chọn thời gian train thấp hơn. Bảng cũng chỉ ra model chính xác nhất và model nhanh nhất để thấy sự đánh đổi.

Không xếp hạng các lượt chưa đủ 30 epoch. Đây là kết quả validation sơ bộ; test dành cho vòng train chính.

In [ ]:
def choose_model(dataset,reports):
    group = [r for r in reports if r["dataset"] == dataset and complete_pilot(r)]
    if len(group) != len(MODEL_SIZES) or {r["model"] for r in group} != {model_name(s) for s in MODEL_SIZES}:
        return None
    best = max(group,key=lambda r:r["validation"]["mask"]["map50_95"])
    eligible = [r for r in group if r["validation"]["mask"]["map50_95"]
                >= best["validation"]["mask"]["map50_95"]-ACCURACY_GAP-1e-12]
    chosen = min(eligible,key=lambda r:(r["benchmark"]["mean_ms"],r["train_wall_seconds"],
                                      -r["validation"]["mask"]["map50_95"]))
    fastest = min(group,key=lambda r:r["benchmark"]["mean_ms"])
    return {"recommended":chosen["model"], "best_accuracy":best["model"], "fastest":fastest["model"]}

def comparison_table(reports):
    rows = []
    for report in reports:
        ready = complete_pilot(report)
        mask = report.get("validation",{}).get("mask",{}) if ready else {}
        bench = report.get("benchmark",{}) if ready else {}
        rows.append({"Dataset":report["dataset"],"Model":report["model"],
                     "Trạng thái":"Hoàn tất" if ready else report["status"],
                     "Epoch":report.get("epochs_completed",0),
                     "P (%)":mask.get("precision",np.nan)*100,
                     "R (%)":mask.get("recall",np.nan)*100,
                     "mAP50 (%)":mask.get("map50",np.nan)*100,
                     "mAP50-95 (%)":mask.get("map50_95",np.nan)*100,
                     "Train (phút)":report.get("train_wall_seconds",np.nan)/60 if ready else np.nan,
                     "Mean (ms/ảnh)":bench.get("mean_ms",np.nan),
                     "P95 (ms/ảnh)":bench.get("p95_ms",np.nan),"FPS":bench.get("fps",np.nan)})
    return pd.DataFrame(rows)

def comparison_figure(table,dataset):
    valid = table[(table["Dataset"] == dataset) & (table["Trạng thái"] == "Hoàn tất")]
    if valid.empty:
        return None
    fig,axes = plt.subplots(1,2,figsize=(11,4))
    for _,row in valid.iterrows():
        axes[0].scatter(row["Mean (ms/ảnh)"],row["mAP50-95 (%)"],s=65)
        axes[0].annotate(row["Model"],(row["Mean (ms/ảnh)"],row["mAP50-95 (%)"]),
                         xytext=(5,6),textcoords="offset points")
    axes[0].set(xlabel="Suy luận: ms/ảnh (thấp hơn tốt hơn)",ylabel="Mask mAP50–95 (%)",title=f"{dataset}: độ chính xác và tốc độ")
    axes[0].margins(.25);axes[0].grid(alpha=.2)
    axes[1].barh(valid["Model"],valid["Train (phút)"],color="#298d92")
    axes[1].set(xlabel="Thời gian train (phút)",title=f"{dataset}: chi phí huấn luyện")
    fig.tight_layout()
    return fig

In [ ]:
PILOT_REPORTS = [get_pilot(dataset,size) for dataset in DATASETS for size in MODEL_SIZES]
COMPARISON = comparison_table(PILOT_REPORTS)
SELECTED = {dataset:choose_model(dataset,PILOT_REPORTS) for dataset in DATASETS}
display(COMPARISON.round(3).fillna("Chưa có"))

for dataset in DATASETS:
    print(dataset, ":", SELECTED[dataset] or "Chờ đủ ba model 30 epoch để chọn.")
    figure = comparison_figure(COMPARISON,dataset)
    if figure is not None:
        plt.show()

In [ ]:
def export_evaluation():
    reports = [get_pilot(dataset,size) for dataset in DATASETS for size in MODEL_SIZES]
    COMPARISON = comparison_table(reports)
    SELECTED = {dataset:choose_model(dataset,reports) for dataset in DATASETS}
    sections = []
    for dataset in DATASETS:
        selected = SELECTED[dataset]
        decision = (f"Đề xuất: {selected['recommended']} · mAP cao nhất: {selected['best_accuracy']} · "
                    f"nhanh nhất: {selected['fastest']}") if selected else "Chưa đủ ba kết quả hợp lệ để đề xuất model."
        table = COMPARISON[COMPARISON["Dataset"] == dataset]
        section = f"<h2>{escape(dataset)}</h2><p>{escape(decision)}</p>" + '<div class="table-wrap">' + table.to_html(index=False,float_format=lambda x:f"{x:.3f}",na_rep="Chưa có") + "</div>"
        figure = comparison_figure(COMPARISON,dataset)
        if figure is not None:
            buffer = io.BytesIO()
            figure.savefig(buffer,format="png",dpi=120,bbox_inches="tight")
            plt.close(figure)
            section += '<img alt="So sánh model" src="data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode() + '">'
        sections.append(section)

    html = """<!doctype html><html lang="vi"><meta charset="utf-8">
    <title>So sánh SEG — thử ba model, chọn một</title>
    <style>body{font:15px system-ui;margin:36px auto;max-width:1250px;padding:0 20px;color:#173b42}
    .table-wrap{overflow-x:auto}h1,h2{color:#126570}table{border-collapse:collapse;width:100%;font-size:13px}
    th,td{border:1px solid #d5e1e4;padding:9px;text-align:right}th{background:#edf5f5}
    img{max-width:100%;height:auto}.note{background:#f3f7f7;padding:16px}</style>
    <h1>So sánh mô hình SEG</h1>"""
    html += f"<p>Pilot {PILOT_EPOCHS} epoch/model · vòng chính {MAIN_EPOCHS} epoch/model · cập nhật {now()}</p>"
    html += f'<p class="note">So sánh riêng từng dataset bằng mask metrics trên validation. Train dùng batch n/s/m tương ứng {BATCH["n"]}/{BATCH["s"]}/{BATCH["m"]} trên {escape(ENVIRONMENT["gpu"])}. Benchmark batch 1, FP16, có warm-up và đồng bộ GPU, không gồm đọc ảnh từ đĩa. Chưa có số liệu được hiển thị là Chưa có, không phải điểm 0. Tiêu chí cân bằng: model nhanh nhất trong nhóm cách mAP50–95 tốt nhất tối đa {ACCURACY_GAP*100:g} điểm phần trăm. Kết quả pilot cần kiểm chứng ở vòng train chính.</p>'
    html += "".join(sections)
    main_rows, validation_rows = [], []
    for dataset, selected in SELECTED.items():
        if not selected:
            continue
        size = next(s for s in MODEL_SIZES if model_name(s) == selected["recommended"])
        report = read_json(own_run(dataset,size,f"main{MAIN_EPOCHS}") / "report.json",{})
        test = report.get("test",{}).get("mask",{}) if report.get("status") == "completed" else {}
        bench = report.get("benchmark",{}) if report.get("status") == "completed" else {}
        pilot = get_pilot(dataset,size)
        if report.get("status") == "completed" and complete_pilot(pilot):
            before = pilot["validation"]["mask"]["map50_95"]
            after = report["validation"]["mask"]["map50_95"]
            validation_rows.append({"Dataset":dataset,"Model":selected["recommended"],
                                    "Epoch chính":report["epochs_completed"],
                                    "Val mAP50-95 pilot (%)":before*100,
                                    "Val mAP50-95 chính (%)":after*100,
                                    "Tăng (điểm %)":(after-before)*100,
                                    "Train pilot (phút)":pilot["train_wall_seconds"]/60,
                                    "Train chính (phút)":report["train_wall_seconds"]/60})
        main_rows.append({"Dataset":dataset,"Model":selected["recommended"],
                          "Trạng thái":report.get("status","pending"),
                          "Epoch":f"{report.get('epochs_completed',0)}/{MAIN_EPOCHS}",
                          "Test P (%)":test.get("precision",np.nan)*100,
                          "Test R (%)":test.get("recall",np.nan)*100,
                          "Test mAP50 (%)":test.get("map50",np.nan)*100,
                          "Test mAP50-95 (%)":test.get("map50_95",np.nan)*100,
                          "Train (phút)":report.get("train_wall_seconds",np.nan)/60,
                          "Mean (ms/ảnh)":bench.get("mean_ms",np.nan),"FPS":bench.get("fps",np.nan)})
    if main_rows:
        html += f"<h2>Vòng chính: tối đa {MAIN_EPOCHS} epoch</h2>"
        html += '<p>Chỉ một model được chọn cho mỗi bài, bắt đầu từ pretrained; patience 20. Chọn checkpoint bằng validation, báo cáo test riêng ở cuối. Số liệu test chưa hoàn tất để trống.</p>'
        html += '<div class="table-wrap">' + pd.DataFrame(main_rows).to_html(index=False,float_format=lambda x:f"{x:.3f}",na_rep="Chưa có") + '</div>'
    if validation_rows:
        html += '<h2>Pilot → vòng chính: so sánh trên cùng tập validation</h2>'
        html += '<p>Bảng này đánh giá lại best.pt bằng cùng quy trình validation. Kết quả test ở bảng trên dùng để báo cáo cuối, không dùng để chọn model.</p>'
        html += '<div class="table-wrap">' + pd.DataFrame(validation_rows).to_html(index=False,float_format=lambda x:f"{x:.3f}",na_rep="Chưa có") + '</div>'
    html += "</html>"
    EVALUATION_FILE.parent.mkdir(parents=True,exist_ok=True)
    EVALUATION_FILE.write_text(html,encoding="utf-8")
    display(FileLink(str(EVALUATION_FILE)))
    print("Báo cáo:",EVALUATION_FILE)
    return EVALUATION_FILE

export_evaluation()

## 7. Train chính — chỉ model được chọn cho mỗi bài

Sau khi xem bảng so sánh, đặt `RUN_MAIN_100 = True` trong ô cấu hình rồi chạy ô này. Ô này gọi train **một lần cho crack và một lần cho pothole**, không train cả ba cỡ ở vòng 100 epoch. Model có thể khác nhau giữa hai bài.

100 epoch là một lịch train mới từ pretrained; dùng validation để chọn checkpoint và đánh giá test riêng ở cuối. Patience 20 có thể kết thúc sớm.

In [ ]:
MAIN_RESULTS = {}
if not RUN_MAIN_100:
    print("Chưa bật vòng chính. Đặt RUN_MAIN_100=True sau khi xem đủ kết quả pilot.")
elif background_job():
    print(background_job(), "— xem báo cáo để theo dõi vòng train chính.")
else:
    PILOT_REPORTS = [get_pilot(dataset,size) for dataset in DATASETS for size in MODEL_SIZES]
    SELECTED = {dataset:choose_model(dataset,PILOT_REPORTS) for dataset in DATASETS}
    if not all(SELECTED.values()):
        print("Cần đủ ba pilot hợp lệ cho mỗi dataset trước khi train chính.")
    else:
        for dataset,selection in SELECTED.items():
            size = next(s for s in MODEL_SIZES if model_name(s) == selection["recommended"])
            print("VÒNG CHÍNH:",dataset,selection["recommended"])
            MAIN_RESULTS[dataset] = train_one(dataset,size,f"main{MAIN_EPOCHS}")
            export_evaluation()
        display(pd.DataFrame([{
            "Dataset":dataset,"Model":r["model"],"Epoch":r["epochs_completed"],
            "Test mAP50":r["test"]["mask"]["map50"],
            "Test mAP50-95":r["test"]["mask"]["map50_95"],
            "Test P":r["test"]["mask"]["precision"],"Test R":r["test"]["mask"]["recall"],
        } for dataset,r in MAIN_RESULTS.items()]))

## 8. Đo chiều dài và diện tích trên ảnh

Đo trên mask ở kích thước ảnh gốc. Tổng diện tích lấy hợp các mask `Pothole`; chiều dài lấy skeleton của hợp các mask `crack`. Đơn vị **px / px²**, chưa có quy đổi mét/m². Với video, áp dụng theo từng frame và không cộng lặp thành chiều dài đoạn đường.

In [ ]:
def skeleton_length(mask):
    pixels = skeletonize(np.asarray(mask,dtype=bool))
    h = np.count_nonzero(pixels[:,:-1] & pixels[:,1:])
    v = np.count_nonzero(pixels[:-1,:] & pixels[1:,:])
    dr = pixels[:-1,:-1] & pixels[1:,1:] & ~pixels[:-1,1:] & ~pixels[1:,:-1]
    dl = pixels[:-1,1:] & pixels[1:,:-1] & ~pixels[:-1,:-1] & ~pixels[1:,1:]
    return float(h+v+math.sqrt(2)*(np.count_nonzero(dr)+np.count_nonzero(dl))),pixels

def measure_result(result,dataset):
    shape = tuple(result.orig_shape)
    union = np.zeros(shape,dtype=bool)
    if result.masks is not None:
        masks = result.masks.data.cpu().numpy() > .5
        if tuple(masks.shape[-2:]) != shape:
            raise ValueError("Mask khác độ phân giải ảnh gốc")
        classes = result.boxes.cls.cpu().numpy().astype(int)
        for mask,cls in zip(masks,classes):
            if dataset == "crack" or result.names[int(cls)].strip().casefold() == "pothole":
                union |= mask
    if dataset == "crack":
        length,skeleton = skeleton_length(union)
        return {"length_px":length},skeleton
    return {"area_px2":int(union.sum())},None

def show_measurement(dataset,image_path,checkpoint):
    with exclusive_gpu():
        model = YOLO(str(checkpoint),task="segment")
        result = model.predict(str(image_path),imgsz=IMGSZ,conf=.25,iou=.7,
                               device=0,half=True,retina_masks=True,verbose=False)[0]
        values,skeleton = measure_result(result,dataset)
        overlay = result.plot()
        if skeleton is not None:
            overlay[skeleton] = (0,0,255)
        output = ROOT / "reports/measurements" / f"main{MAIN_EPOCHS}" / dataset
        output.mkdir(parents=True,exist_ok=True)
        cv2.imwrite(str(output / "example.png"),overlay)
        write_json(output / "example.json",dict(dataset=dataset,image=str(image_path),
                   checkpoint=str(checkpoint),checkpoint_sha256=file_sha256(checkpoint),
                   measurements=values,image_shape=list(result.orig_shape),
                   units="pixels; chưa hiệu chuẩn sang mét/m²",created_at=now()))
        plt.figure(figsize=(9,7));plt.imshow(cv2.cvtColor(overlay,cv2.COLOR_BGR2RGB))
        plt.title(f"{dataset}: {values}");plt.axis("off");plt.show()
        del model,result
        gc.collect();torch.cuda.empty_cache()
        return values

In [ ]:
if background_job():
    print(background_job(), "— chưa chạy demo GPU để tránh ảnh hưởng training.")
else:
    for dataset in DATASETS:
        selection = SELECTED.get(dataset)
        if not selection:
            print(dataset,": chờ chọn model.")
            continue
        size = next(s for s in MODEL_SIZES if model_name(s) == selection["recommended"])
        final = read_json(own_run(dataset,size,f"main{MAIN_EPOCHS}") / "report.json")
        chosen = final if final and final.get("status") == "completed" else get_pilot(dataset,size)
        checkpoint = chosen.get("checkpoint",{}).get("best") or chosen.get("validation",{}).get("checkpoint")
        if checkpoint:
            sample_image = SAMPLES[dataset]["images"][0]
            print("Ảnh:",sample_image)
            show_measurement(dataset,sample_image,checkpoint)

## Cách sử dụng ngắn gọn

1. Chạy từ đầu đến phần so sánh. Khi hàng đợi cũ còn chạy, xem tiến độ và chạy lại phần so sánh sau khi các lượt kết thúc.
2. Xem mAP/P/R, thời gian train và latency/FPS; mỗi bài có một model được đề xuất.
3. Bật `RUN_MAIN_100=True`, chạy ô train chính để tập trung vào hai model được chọn, mỗi bài một model.
4. Xem kết quả test và demo đo pixel. Các notebook SEG tách riêng trước đó được lưu trong archive; file này là notebook làm việc chính.